In [1]:
# ============================================================
# exp_20260426_057_xgb046b_high_interaction_min
# Purpose:
#   046b backbone-ish XGB に High-targeted interaction 最小セットを追加
#   raw / biased CV, OOF/pred保存, 既存主力との相関確認
# ============================================================


In [2]:
import os
import json
import random
import warnings
from pathlib import Path
import itertools

import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import balanced_accuracy_score
import torch

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

In [3]:
class CFG:
    EXP_ID = "exp_20260426_057_xgb046b_high_interaction_min"
    EXP_NAME = "xgb046b_high_interaction_min"
    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"

    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"
    SEED = 2026
    N_FOLDS = 5
    NUM_CLASSES = 3

    XGB_PARAMS = {
        "max_depth": 4,
        "colsample_bytree": 0.8,
        "subsample": 0.8,
        "n_estimators": 512,
        "learning_rate": 0.1,
        "early_stopping_rounds": 100,
        "random_state": SEED,
        "n_jobs": -1,
        "enable_categorical": True,
        "alpha": 5,
        "reg_lambda": 5,
        "max_leaves": 30,
        "min_child_weight": 2,
        "tree_method": "hist",
        "max_bin": 10000,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }

    best_params_path = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/exp_20260415_046_xgb024_optuna/best_params.json"
    with open(best_params_path, mode="rt", encoding="utf-8") as f:
        best_params_payload = json.load(f)

    XGB_PARAMS.update(best_params_payload["best_params"])
    XGB_PARAMS["n_estimators"] = 2500

In [4]:
def seed_everything(seed):
    np.random.seed(seed)
    random.seed(seed)

def balanced_acc_score_mc(y_true, y_pred):
    if len(y_pred.shape) == 2:
        y_pred = np.argmax(y_pred, axis=1)
    c = len(np.unique(y_true))
    acc = 0.0
    for i in range(c):
        acc += np.sum((y_true == i) & (y_pred == i)) / np.sum(y_true == i) / c
    return acc

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def safe_logit_like(proba, eps=1e-12):
    return np.log(np.clip(proba, eps, 1.0))

def apply_bias(proba, bias):
    """
    bias: iterable of len 3
    apply in log-prob space, then renormalize
    """
    logp = safe_logit_like(proba)
    adjusted = logp + np.array(bias, dtype=np.float32).reshape(1, -1)

    adjusted = adjusted - adjusted.max(axis=1, keepdims=True)
    expv = np.exp(adjusted)
    out = expv / expv.sum(axis=1, keepdims=True)
    return out.astype(np.float32)

def score_bias(proba, y_true, bias):
    adj = apply_bias(proba, bias)
    pred = np.argmax(adj, axis=1)
    return balanced_accuracy_score(y_true, pred)

def run_grid_search(proba, y_true, bias_ranges):
    best_bias = None
    best_score = -1.0

    for b0, b1, b2 in itertools.product(*bias_ranges):
        bias = (b0, b1, b2)
        sc = score_bias(proba, y_true, bias)
        if sc > best_score:
            best_score = sc
            best_bias = bias

    return best_bias, float(best_score)


seed_everything(CFG.SEED)

In [5]:
drop_cols = [CFG.ID_COL]

train = pd.read_csv(CFG.TRAIN_PATH).drop(drop_cols, axis=1)
test = pd.read_csv(CFG.TEST_PATH).drop(drop_cols, axis=1)

print(f"train.shape={train.shape}, test.shape={test.shape}")

target2idx = {v: i for i, v in enumerate(train[CFG.TARGET_COL].unique())}
idx2target = {v: k for k, v in target2idx.items()}

train[CFG.TARGET_COL] = train[CFG.TARGET_COL].map(target2idx)

CATS = [c for c in test.columns if train[c].dtype == object]
NUMS = [c for c in test.columns if c not in CATS]

print("CATS:", CATS)
print("NUMS:", NUMS)
display(train.head())

train.shape=(630000, 20), test.shape=(270000, 19)
CATS: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
NUMS: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']


,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,0
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,0
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,0
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,1
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,0


In [6]:
# digit FE
M = train[NUMS].max()

def add_digit_features(df):
    df = df.copy()

    for c in NUMS:
        for k in range(-4, 4):
            df[f"{c}_digit{k}"] = (df[c] // (10 ** k) % 10).astype("int8")

        if M[c] < 10:
            df[c] = df[c].round(3)
        elif M[c] < 100:
            df[c] = df[c].round(2)
        else:
            df[c] = df[c].round(1)

    return df

In [7]:
def add_high_interaction_features(df):
    df = df.copy()

    df["moisture_low"] = (df["Soil_Moisture"] < 25).astype("int8")
    df["rain_low"] = (df["Rainfall_mm"] < 300).astype("int8")
    df["temp_high"] = (df["Temperature_C"] > 30).astype("int8")
    df["wind_high"] = (df["Wind_Speed_kmh"] > 10).astype("int8")

    df["moisture_low_x_stage"] = (
        df["moisture_low"].astype(str) + "_" + df["Crop_Growth_Stage"].astype(str)
    )

    df["rain_low_x_stage"] = (
        df["rain_low"].astype(str) + "_" + df["Crop_Growth_Stage"].astype(str)
    )

    df["temp_high_x_stage"] = (
        df["temp_high"].astype(str) + "_" + df["Crop_Growth_Stage"].astype(str)
    )

    df["wind_high_x_mulching"] = (
        df["wind_high"].astype(str) + "_" + df["Mulching_Used"].astype(str)
    )

    df["irrigation_x_source"] = (
        df["Irrigation_Type"].astype(str) + "_" + df["Water_Source"].astype(str)
    )

    df["crop_x_stage"] = (
        df["Crop_Type"].astype(str) + "_" + df["Crop_Growth_Stage"].astype(str)
    )

    df["moisture_deficit"] = np.maximum(0, 25 - df["Soil_Moisture"]).astype("float32")
    df["rain_deficit"] = np.maximum(0, 300 - df["Rainfall_mm"]).astype("float32")
    df["temp_excess"] = np.maximum(0, df["Temperature_C"] - 30).astype("float32")
    df["wind_excess"] = np.maximum(0, df["Wind_Speed_kmh"] - 10).astype("float32")

    df["high_pressure_score"] = (
        df["moisture_low"] + df["rain_low"] + df["temp_high"] + df["wind_high"]
    ).astype("int8")

    return df

In [8]:
train = add_digit_features(train)
test = add_digit_features(test)

train = add_high_interaction_features(train)
test = add_high_interaction_features(test)

In [9]:
DROP = [c for c in test.columns if test[c].nunique() == 1]
print("DROP:", DROP)

train.drop(DROP, axis=1, inplace=True)
test.drop(DROP, axis=1, inplace=True)

DROP: ['Soil_pH_digit1', 'Soil_pH_digit2', 'Soil_pH_digit3', 'Soil_Moisture_digit2', 'Soil_Moisture_digit3', 'Organic_Carbon_digit1', 'Organic_Carbon_digit2', 'Organic_Carbon_digit3', 'Electrical_Conductivity_digit1', 'Electrical_Conductivity_digit2', 'Electrical_Conductivity_digit3', 'Temperature_C_digit2', 'Temperature_C_digit3', 'Humidity_digit2', 'Humidity_digit3', 'Sunlight_Hours_digit2', 'Sunlight_Hours_digit3', 'Wind_Speed_kmh_digit2', 'Wind_Speed_kmh_digit3', 'Field_Area_hectare_digit2', 'Field_Area_hectare_digit3', 'Previous_Irrigation_mm_digit3']


## category remap

In [10]:
# category remap
HIGH_CATS = [
    "moisture_low_x_stage",
    "rain_low_x_stage",
    "temp_high_x_stage",
    "wind_high_x_mulching",
    "irrigation_x_source",
    "crop_x_stage",
    "high_pressure_score",
]

HIGH_NUMS = [
    "moisture_low",
    "rain_low",
    "temp_high",
    "wind_high",
    "moisture_deficit",
    "rain_deficit",
    "temp_excess",
    "wind_excess",
]

CATEGORY = CATS + [c for c in test.columns if "digit" in c] + HIGH_CATS
CATEGORY = list(dict.fromkeys(CATEGORY))

for c in CATEGORY:
    freq = train[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)

    train[c] = train[c].map(lambda x: mapping.get(x, mapping_default))
    test[c] = test[c].map(lambda x: mapping.get(x, mapping_default))

FEATURES = CATEGORY + NUMS + HIGH_NUMS
FEATURES = list(dict.fromkeys(FEATURES))

DROP_ORIGINAL_CATS = CATS + HIGH_CATS

print("n_CATEGORY =", len(CATEGORY))
print("n_FEATURES =", len(FEATURES))
display(train[FEATURES + [CFG.TARGET_COL]].head())

n_CATEGORY = 81
n_FEATURES = 100


,Soil_Type,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Mulching_Used,Region,Soil_pH_digit-4,Soil_pH_digit-3,Soil_pH_digit-2,Soil_pH_digit-1,Soil_pH_digit0,Soil_Moisture_digit-4,Soil_Moisture_digit-3,Soil_Moisture_digit-2,Soil_Moisture_digit-1,Soil_Moisture_digit0,Soil_Moisture_digit1,Organic_Carbon_digit-4,Organic_Carbon_digit-3,Organic_Carbon_digit-2,Organic_Carbon_digit-1,Organic_Carbon_digit0,Electrical_Conductivity_digit-4,Electrical_Conductivity_digit-3,Electrical_Conductivity_digit-2,Electrical_Conductivity_digit-1,Electrical_Conductivity_digit0,Temperature_C_digit-4,Temperature_C_digit-3,Temperature_C_digit-2,Temperature_C_digit-1,Temperature_C_digit0,Temperature_C_digit1,Humidity_digit-4,Humidity_digit-3,Humidity_digit-2,Humidity_digit-1,Humidity_digit0,Humidity_digit1,Rainfall_mm_digit-4,Rainfall_mm_digit-3,Rainfall_mm_digit-2,Rainfall_mm_digit-1,Rainfall_mm_digit0,Rainfall_mm_digit1,Rainfall_mm_digit2,Rainfall_mm_digit3,Sunlight_Hours_digit-4,Sunlight_Hours_digit-3,Sunlight_Hours_digit-2,Sunlight_Hours_digit-1,Sunlight_Hours_digit0,Sunlight_Hours_digit1,Wind_Speed_kmh_digit-4,Wind_Speed_kmh_digit-3,Wind_Speed_kmh_digit-2,Wind_Speed_kmh_digit-1,Wind_Speed_kmh_digit0,Wind_Speed_kmh_digit1,Field_Area_hectare_digit-4,Field_Area_hectare_digit-3,Field_Area_hectare_digit-2,Field_Area_hectare_digit-1,Field_Area_hectare_digit0,Field_Area_hectare_digit1,Previous_Irrigation_mm_digit-4,Previous_Irrigation_mm_digit-3,Previous_Irrigation_mm_digit-2,Previous_Irrigation_mm_digit-1,Previous_Irrigation_mm_digit0,Previous_Irrigation_mm_digit1,Previous_Irrigation_mm_digit2,moisture_low_x_stage,rain_low_x_stage,temp_high_x_stage,wind_high_x_mulching,irrigation_x_source,crop_x_stage,high_pressure_score,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Field_Area_hectare,Previous_Irrigation_mm,moisture_low,rain_low,temp_high,wind_high,moisture_deficit,rain_deficit,temp_excess,wind_excess,Irrigation_Need
0,2,0,3,2,3,3,0,2,0,0,4,1,4,0,0,5,6,2,3,0,0,5,9,1,0,0,2,9,3,0,0,5,4,3,2,0,0,0,2,9,1,0,0,2,2,4,6,6,1,1,1,8,0,0,0,0,0,3,3,7,0,0,0,8,3,4,0,0,0,0,6,3,0,1,1,3,2,0,15,19,0,4.92,32.58,1.01,3.05,15.01,50.61,726.0,5.90,16.79,0.82,112.2,0,0,0,1,0.00,0.0,0.0,6.79,0
1,1,4,2,0,2,1,1,0,0,0,8,2,2,0,0,1,3,7,1,0,0,6,0,0,0,0,9,4,0,1,1,2,1,0,1,0,0,4,9,3,3,0,0,0,0,4,8,9,1,1,1,2,0,6,0,0,1,9,1,1,1,0,0,1,6,6,0,0,0,0,6,5,5,0,3,2,1,2,11,7,2,7.08,56.61,0.44,2.00,22.92,67.86,985.7,6.98,3.39,5.27,47.2,0,0,0,0,0.00,0.0,0.0,0.00,0
2,1,1,2,0,1,0,1,4,1,1,9,0,1,0,1,9,7,8,2,1,1,8,6,0,0,1,4,8,0,0,0,7,1,9,1,0,0,1,1,5,6,0,0,3,0,1,9,3,2,0,0,1,3,6,0,0,1,4,0,1,1,0,1,6,6,8,0,0,0,6,8,2,0,1,3,2,1,2,5,9,2,5.69,27.71,0.81,2.83,26.97,92.22,2201.7,6.05,3.85,8.24,110.4,0,0,0,0,0.00,0.0,0.0,0.00,0
3,0,4,1,0,0,1,1,0,1,1,5,0,1,0,1,2,0,5,4,1,1,6,2,1,0,0,5,8,2,0,1,2,6,8,2,0,0,9,6,0,3,0,0,9,4,3,3,2,0,0,0,4,5,2,0,0,1,0,1,5,1,0,1,5,4,8,0,0,1,0,9,4,3,0,5,1,3,2,1,21,0,5.65,13.32,1.33,0.87,13.32,61.57,1357.3,9.12,2.31,8.32,53.8,1,0,0,0,11.68,0.0,0.0,0.00,1
4,1,4,3,1,0,1,0,0,0,0,5,1,2,0,0,4,2,4,1,0,0,0,2,0,0,0,3,4,2,0,0,1,9,1,1,0,0,0,8,0,6,0,1,1,8,5,2,8,0,0,1,3,0,6,0,0,0,2,5,1,0,0,0,1,4,7,0,0,0,9,6,4,4,0,1,3,2,0,1,22,0,7.96,59.14,0.38,0.96,20.22,91.11,1538.2,6.95,13.94,7.37,93.2,0,0,0,1,0.00,0.0,0.0,3.94,0


## class weight

In [11]:
unique, counts = np.unique(train[CFG.TARGET_COL].values, return_counts=True)
count_dict = dict(zip(unique, counts))
avg_count = len(train) / len(unique)
weights_dict = {cls: avg_count / cnt for cls, cnt in count_dict.items()}
sample_weights = np.array([weights_dict[y] for y in train[CFG.TARGET_COL]])

print("weights_dict =", weights_dict)

weights_dict = {np.int64(0): np.float64(0.5676949153458749), np.int64(1): np.float64(0.8783891180136694), np.int64(2): np.float64(9.995716121662145)}


## OrderedTE class

In [12]:
class OrderedTE:
    def __init__(self, a=1):
        self.a = a

    def fit(self, train, category_cols=None, target_col="target"):
        if category_cols is None:
            category_cols = []

        self.train = train.copy()
        self.target_col = target_col
        self.category_cols = category_cols

        self.classes_ = sorted(train[target_col].unique())
        self.num_classes_ = len(self.classes_)
        self.global_prior_ = train[target_col].value_counts(normalize=True).sort_index().values

        for c in self.category_cols:
            stats_list = []

            for k, cls in enumerate(self.classes_):
                y_binary = (train[target_col] == cls).astype(int)

                df = train[[c]].copy()
                df["y"] = y_binary.values
                df["cnt"] = 1

                df["cum_cnt"] = df.groupby(c)["cnt"].cumsum() - df["cnt"]
                df["cum_sum"] = df.groupby(c)["y"].cumsum() - df["y"]

                smooth_prior = self.a * self.global_prior_[k]

                te_col = f"{c}_TE_cls{cls}"
                df[te_col] = (df["cum_sum"] + smooth_prior) / (df["cum_cnt"] + self.a)
                df.loc[df["cum_cnt"] == 0, te_col] = self.global_prior_[k]

                self.train[te_col] = df[te_col].values

                stats_df = df.groupby(c)["y"].agg(["count", "sum"]).reset_index()
                stats_df.columns = [c, f"{c}_count_cls{cls}", f"{c}_sum_cls{cls}"]
                stats_df[f"{c}_prior_cls{cls}"] = self.global_prior_[k]
                stats_list.append(stats_df)

            combined_stats = stats_list[0]
            for i in range(1, len(stats_list)):
                combined_stats = combined_stats.merge(stats_list[i], on=c, how="outer")
            setattr(self, f"{c}_stats", combined_stats)

        return self.train

    def transform(self, test):
        test = test.copy()

        for c in self.category_cols:
            stats_df = getattr(self, f"{c}_stats")
            test = test.merge(stats_df, on=c, how="left")

            for k, cls in enumerate(self.classes_):
                te_col = f"{c}_TE_cls{cls}"
                count_col = f"{c}_count_cls{cls}"
                sum_col = f"{c}_sum_cls{cls}"
                prior_col = f"{c}_prior_cls{cls}"

                if count_col in test.columns:
                    test[te_col] = (test[sum_col] + self.a * test[prior_col]) / (test[count_col] + self.a)
                    test[te_col] = test[te_col].fillna(test[prior_col])
                    test.drop([count_col, sum_col, prior_col], axis=1, inplace=True)
                else:
                    test[te_col] = self.global_prior_[k]

        return test

## CV main

In [13]:
X = train.drop([CFG.TARGET_COL], axis=1)
y = train[CFG.TARGET_COL]
test_X = test.copy()

oof_raw = np.zeros((len(y), CFG.NUM_CLASSES))
pred_raw = np.zeros((len(test_X), CFG.NUM_CLASSES))

kf = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=42)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    print(f"\nFold {fold}/{CFG.N_FOLDS}")

    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()
    train_weights = sample_weights[train_idx]

    te = OrderedTE(a=1)
    full_df = pd.concat((X_train, y_train), axis=1)
    full_df["weight"] = train_weights

    # 共有コードでは4回増殖しているが、今回は最小再現なので1回だけ
    te_train = te.fit(full_df.sample(frac=1, random_state=42), category_cols=FEATURES, target_col=CFG.TARGET_COL)

    X_train = te_train.drop([CFG.TARGET_COL, "weight"], axis=1)
    y_train = te_train[CFG.TARGET_COL]
    train_weights = te_train["weight"].values

    X_val = te.transform(X_val)
    X_test = te.transform(test_X)

    # 元カテゴリは落とす
    DROP_ORIGINAL_CATS = CATS + HIGH_CATS
    
    X_train.drop(DROP_ORIGINAL_CATS, axis=1, inplace=True, errors="ignore")
    X_val.drop(DROP_ORIGINAL_CATS, axis=1, inplace=True, errors="ignore")
    X_test.drop(DROP_ORIGINAL_CATS, axis=1, inplace=True, errors="ignore")

    params = CFG.XGB_PARAMS.copy()
    if not torch.cuda.is_available():
        params["device"] = "cpu"

    model = XGBClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        sample_weight=train_weights,
        verbose=100,
    )

    y_pred = model.predict_proba(X_val)
    oof_raw[val_idx] = y_pred
    pred_raw += model.predict_proba(X_test) / CFG.N_FOLDS

    fold_ba = balanced_acc_score_mc(y_val.values, y_pred)
    fold_scores.append(float(fold_ba))
    print(f"fold_ba = {fold_ba:.6f}")


Fold 1/5
[0]	validation_0-mlogloss:1.03834
[100]	validation_0-mlogloss:0.10401
[200]	validation_0-mlogloss:0.07070
[300]	validation_0-mlogloss:0.06169
[400]	validation_0-mlogloss:0.05800
[500]	validation_0-mlogloss:0.05556
[600]	validation_0-mlogloss:0.05411
[700]	validation_0-mlogloss:0.05301
[800]	validation_0-mlogloss:0.05223
[900]	validation_0-mlogloss:0.05151
[1000]	validation_0-mlogloss:0.05094
[1100]	validation_0-mlogloss:0.05039
[1200]	validation_0-mlogloss:0.05003
[1300]	validation_0-mlogloss:0.04961
[1400]	validation_0-mlogloss:0.04926
[1500]	validation_0-mlogloss:0.04903
[1600]	validation_0-mlogloss:0.04881
[1700]	validation_0-mlogloss:0.04854
[1800]	validation_0-mlogloss:0.04828
[1900]	validation_0-mlogloss:0.04811
[2000]	validation_0-mlogloss:0.04790
[2100]	validation_0-mlogloss:0.04775
[2200]	validation_0-mlogloss:0.04760
[2300]	validation_0-mlogloss:0.04740
[2400]	validation_0-mlogloss:0.04730
[2499]	validation_0-mlogloss:0.04722
fold_ba = 0.976055

Fold 2/5
[0]	validat

## raw CV

In [14]:
raw_cv_ba = balanced_acc_score_mc(y.values, oof_raw)
print(f"raw_cv_ba = {raw_cv_ba:.6f}")

raw_cv_ba = 0.977587


## bias search

In [15]:
# ---------------------------------------------------------
# Stage 1: coarse
# ---------------------------------------------------------
coarse_vals = np.arange(-1.0, 1.01, 0.1)
best_bias_1, best_score_1 = run_grid_search(
    oof_raw, y,
    [coarse_vals, coarse_vals, coarse_vals]
)
print("stage1 best_bias:", best_bias_1, "score:", best_score_1)

stage1 best_bias: (np.float64(-1.0), np.float64(-1.0), np.float64(0.3999999999999997)) score: 0.9796418506267358


In [16]:
# ---------------------------------------------------------
# Stage 2: local refine
# ---------------------------------------------------------
b0, b1, b2 = best_bias_1
local_vals_0 = np.arange(b0 - 0.10, b0 + 0.1001, 0.02)
local_vals_1 = np.arange(b1 - 0.10, b1 + 0.1001, 0.02)
local_vals_2 = np.arange(b2 - 0.10, b2 + 0.1001, 0.02)

best_bias_2, best_score_2 = run_grid_search(
    oof_raw, y,
    [local_vals_0, local_vals_1, local_vals_2]
)
print("stage2 best_bias:", best_bias_2, "score:", best_score_2)

stage2 best_bias: (np.float64(-1.1), np.float64(-0.98), np.float64(0.3799999999999998)) score: 0.9796726006402646


In [17]:
# ---------------------------------------------------------
# Stage 3: ultra-local refine
# ---------------------------------------------------------
b0, b1, b2 = best_bias_2
ultra_vals_0 = np.arange(b0 - 0.02, b0 + 0.0201, 0.005)
ultra_vals_1 = np.arange(b1 - 0.02, b1 + 0.0201, 0.005)
ultra_vals_2 = np.arange(b2 - 0.02, b2 + 0.0201, 0.005)

best_bias_3, best_score_3 = run_grid_search(
    oof_raw, y,
    [ultra_vals_0, ultra_vals_1, ultra_vals_2]
)
print("stage3 best_bias:", best_bias_3, "score:", best_score_3)

final_bias = best_bias_3
final_cv = best_score_3

oof_biased = apply_bias(oof_raw, final_bias)
pred_biased = apply_bias(pred_raw, final_bias)

np.save(CFG.SAVE_DIR / "oof_xgb046b_high_interaction_min_biased.npy", oof_biased)
np.save(CFG.SAVE_DIR / "pred_xgb046b_high_interaction_min_biased.npy", pred_biased)
np.save(CFG.SAVE_DIR / "best_class_bias_refined.npy", np.array(final_bias, dtype=np.float32))

final_test_preds = np.argmax(pred_biased, axis=1)

submission = pd.read_csv(
    "/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv")
submission[CFG.TARGET_COL] = pd.Series(final_test_preds).map(idx2target)
submission.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

cv_result = {
    "exp_id": CFG.EXP_ID,
    "base_parent": "exp_20260409_024_xgb_digit_orderedte_min",
    "raw_cv": float(raw_cv_ba),
    "refined_biased_cv": float(final_cv),
    "best_bias": list(map(float, final_bias)),
    "search": {
        "stage1": {"range": [-1.0, 1.0], "step": 0.1},
        "stage2": {"center": list(map(float, best_bias_1)), "width": 0.10, "step": 0.02},
        "stage3": {"center": list(map(float, best_bias_2)), "width": 0.02, "step": 0.005},
    }
}
save_json(cv_result, f"{CFG.SAVE_DIR}/cv_result.json")

print("final_bias:", final_bias)
print("final_cv:", final_cv)
print("saved to:", CFG.SAVE_DIR)

stage3 best_bias: (np.float64(-1.1050000000000004), np.float64(-1.0), np.float64(0.36499999999999977)) score: 0.9797185002067851
final_bias: (np.float64(-1.1050000000000004), np.float64(-1.0), np.float64(0.36499999999999977))
final_cv: 0.9797185002067851
saved to: /kaggle/working/exp_20260426_057_xgb046b_high_interaction_min
